In [1]:
# Data Manipulation
import pandas as pd
import numpy as np

# System Performance and Benchmarking
import time
import tracemalloc
import psutil
import os

# Data Export and File Handling
import json
from pathlib import Path

# Cryptographic Operations
import secrets

Implementation of Diffie–Hellman Key Exchange

- It generates a random private key, computes the corresponding public key using modular exponentiation, and derives the shared secret using the other party's public key and its own private key.

In [2]:
class DiffieHellman:

    def __init__(self, p, g):
        self.p = p
        self.g = g

    def generate_private_key(self):
        return secrets.randbelow(self.p - 2) + 2

    def generate_public_key(self, private_key):
        return pow(self.g, private_key, self.p)

    def generate_shared_secret(self, other_public_key, private_key):
        return pow(other_public_key, private_key, self.p)

It defines the key sizes, initializes the corresponding prime modulus values, and sets the generator to be used for the Diffie–Hellman key exchange during benchmarking.

In [3]:
key_sizes = [1000, 2000, 4000, 8000]

safe_primes = {
    1000: (2**1000 - 1),
    2000: (2**2000 - 1),
    4000: (2**4000 - 1),
    8000: (2**8000 - 1)
}

generator = 2

Performs the Diffie–Hellman key exchange for the specified key size by generating private and public keys for two communicating parties, computing the shared secret, and returning the shared secrets established by both parties.

In [4]:
def perform_key_exchange(bit_size):
    p = safe_primes[bit_size]
    g = generator

    dh = DiffieHellman(p, g)

    alice_private = dh.generate_private_key()
    alice_public = dh.generate_public_key(alice_private)

    bob_private = dh.generate_private_key()
    bob_public = dh.generate_public_key(bob_private)

    alice_secret = dh.generate_shared_secret(bob_public, alice_private)
    bob_secret = dh.generate_shared_secret(alice_public, bob_private)

    return alice_secret, bob_secret

Executes the Diffie–Hellman key exchange for each specified key size and verifies whether both communicating parties establish the same shared secret.

In [5]:
for bits in key_sizes:
    alice_secret, bob_secret = perform_key_exchange(bits)

    print(f"{bits}-bit:", alice_secret == bob_secret)

1000-bit: True
2000-bit: True
4000-bit: True
8000-bit: True


Performs multiple Diffie–Hellman key exchange trials across different key sizes and attack conditions, measuring runtime, CPU usage, memory consumption, key generation rate, and successful key establishment. 

It also simulates man-in-the-middle attack scenarios by introducing an attacker-controlled key exchange.

In [10]:
TRIALS = 10

attack_scenarios = [
    ("MITM_0", 0.0),
    ("MITM_25", 0.25),
    ("MITM_50", 0.50),
    ("MITM_100", 1.0)
]

detailed_results = []

for bits in key_sizes:

    p = safe_primes[bits]
    g = generator

    for attack_name, attack_rate in attack_scenarios:

        for trial in range(1, TRIALS + 1):

            tracemalloc.start()

            start_wall = time.perf_counter()
            start_cpu = time.process_time()

            dh = DiffieHellman(p, g)

            # Alice
            a_private = dh.generate_private_key()
            a_public = dh.generate_public_key(a_private)

            # Bob
            b_private = dh.generate_private_key()
            b_public = dh.generate_public_key(b_private)

            # Shared Secret
            if attack_name == "none":

                s1 = dh.generate_shared_secret(b_public, a_private)
                s2 = dh.generate_shared_secret(a_public, b_private)

            else:
            
                if secrets.randbelow(100) < int(attack_rate * 100):
            
                    # Eve creates two separate identities
                    eve_private_a = dh.generate_private_key()
                    eve_public_a = dh.generate_public_key(eve_private_a)
            
                    eve_private_b = dh.generate_private_key()
                    eve_public_b = dh.generate_public_key(eve_private_b)
            
                    # Alice shares secret with Eve
                    s1 = dh.generate_shared_secret(
                        eve_public_a,
                        a_private
                    )
            
                    # Bob shares secret with Eve
                    s2 = dh.generate_shared_secret(
                        eve_public_b,
                        b_private
                    )
            
                else:
            
                    s1 = dh.generate_shared_secret(
                        b_public,
                        a_private
                    )
            
                    s2 = dh.generate_shared_secret(
                        a_public,
                        b_private
                    )

            end_cpu = time.process_time()
            end_wall = time.perf_counter()

            current, peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            peak_ram_mb = peak / (1024 * 1024)

            runtime = end_wall - start_wall
            cpu_time = end_cpu - start_cpu

            shared_secret_bits = bits
            key_gen_rate = shared_secret_bits / runtime if runtime > 0 else 0

            detailed_results.append({
                "protocol": "Diffie-Hellman",
                "key_size": bits,
                "trial": trial,
                "runtime_seconds": runtime,
                "cpu_time_seconds": cpu_time,
                "peak_ram_mb": peak_ram_mb,
                "key_gen_rate": key_gen_rate,
                "attack_scenario": attack_name,
                "attack_rate": attack_rate,
                "success": s1 == s2
            })

print("Benchmark completed.")

Benchmark completed.


It converts benchmark results into a DataFrame and displays the initial records.

In [11]:
detailed_df = pd.DataFrame(detailed_results)

detailed_df.head()

,protocol,key_size,trial,runtime_seconds,cpu_time_seconds,peak_ram_mb,key_gen_rate,attack_scenario,attack_rate,success
0,Diffie-Hellman,1000,1,0.019503,0.015625,0.002975,51275.477313,MITM_0,0.0,True
1,Diffie-Hellman,1000,2,0.018538,0.015625,0.004146,53942.378902,MITM_0,0.0,True
2,Diffie-Hellman,1000,3,0.025831,0.031250,0.003540,38713.324044,MITM_0,0.0,True
3,Diffie-Hellman,1000,4,0.025051,0.015625,0.003605,39918.884729,MITM_0,0.0,True
4,Diffie-Hellman,1000,5,0.026990,0.031250,0.003605,37050.073271,MITM_0,0.0,True


It aggregates benchmark results by key size, calculates statistical measures, and summarizes performance metrics for analysis.

In [12]:
aggregated_df = detailed_df.groupby(
    ["key_size", "attack_scenario"]
).agg({
    "runtime_seconds": ["mean", "std"],
    "cpu_time_seconds": ["mean", "std"],
    "key_gen_rate": ["mean", "std"],
    "peak_ram_mb": ["mean", "std"],
    "success": "sum"
})

aggregated_df.columns = [
    "_".join(col).strip()
    for col in aggregated_df.columns.values
]

aggregated_df = aggregated_df.reset_index()

aggregated_df

,key_size,attack_scenario,runtime_seconds_mean,runtime_seconds_std,cpu_time_seconds_mean,cpu_time_seconds_std,key_gen_rate_mean,key_gen_rate_std,peak_ram_mb_mean,peak_ram_mb_std,success_sum
0,1000,MITM_0,0.022925,0.003493,0.021875,0.008069,44512.751744,6543.530522,0.003628,0.000291,10
1,1000,MITM_100,0.028646,0.002038,0.026562,0.010546,35056.242329,2310.883817,0.004154,0.000210,0
2,1000,MITM_25,0.022236,0.005246,0.023438,0.008235,47783.911424,14226.693225,0.003735,0.000270,9
3,1000,MITM_50,0.025300,0.005851,0.025000,0.010925,41376.998646,8978.320803,0.003890,0.000430,6
4,2000,MITM_0,0.076297,0.004313,0.076563,0.008869,26288.129200,1469.335407,0.006604,0.000474,10
5,2000,MITM_100,0.103033,0.002846,0.096875,0.009882,19424.710855,539.221920,0.007759,0.000413,0
6,2000,MITM_25,0.071552,0.005343,0.071875,0.008069,28118.357757,2485.199195,0.006412,0.000670,10
7,2000,MITM_50,0.080467,0.019126,0.079687,0.020104,26163.698528,6393.534956,0.007491,0.001680,7
8,4000,MITM_0,0.379325,0.022651,0.360938,0.030769,10578.890084,630.126941,0.012729,0.001153,10
9,4000,MITM_100,0.555288,0.015343,0.540625,0.019764,7208.553826,204.570742,0.013998,0.001007,0


It creates a results directory and exports detailed benchmark results in CSV and JSON formats for storage and analysis.

In [14]:
detailed_df.to_csv(
    "dh_detailed_results.csv",
    index=False
)

detailed_df.to_json(
    "dh_detailed_results.json",
    orient="records",
    indent=4
)